# #8.0 Chat UI — 엔진을 직접 돌려보기

**이 노트북의 목적**: #8.0의 *핵심 엔진*(Agent · 메모리 · Runner · 스트리밍)을 주피터에서 한 줄씩 돌려보며 이해한다.

> ⚠️ **Streamlit UI(`st.chat_input`, `st.chat_message` 등)는 주피터에서 안 돈다.** 그건 `streamlit run app.py`로만 뜬다.
> 그래서 여기선 **속(엔진)**만 본다. 화면(Streamlit)은 맨 아래에서 어떻게 얹히는지 *설명*으로 매핑한다.

**오늘 대화에서 잡은 렌즈로 읽기:**
- 에이전트 = 네 **Gem** 그 자체 (`Agent` = 이름 + 요청사항 + 도구)
- LLM은 글자 예측기라 **기억을 못 한다** → 메모리는 `SQLiteSession`으로 **밖에** 쌓는다
- '오늘 마감'이 깨지던 이유(컨텍스트 윈도우)도 여기서 메모리 구조를 직접 보면 감이 온다

## 0) 준비 — API 키 로드

`WebSearchTool`은 안 쓰지만, Agent가 GPT를 부르려면 `OPENAI_API_KEY`가 필요하다.

**처음 한 번만**: 터미널에서 `cp .env.example .env` 후 `.env`에 실제 키를 넣을 것.

> `load_dotenv()`는 *이미 있는* 환경변수는 안 덮어쓴다. 키를 바꿨는데 안 먹으면 커널 재시작 또는 `load_dotenv(override=True)`. (1주차에 삽질했던 그거)

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

key = os.getenv("OPENAI_API_KEY")
print("키 있음 ✅" if key else "키 없음 ❌ — .env 만들어서 OPENAI_API_KEY 넣기")
print("앞 7자리:", key[:7] + "..." if key else "-")

키 있음 ✅
앞 7자리: sk-proj...


In [3]:
from agents import Agent, Runner, SQLiteSession

## 1) Agent 만들기 = 네 Gem 만들기

| Gem 화면 | 여기 코드 |
|---|---|
| 이름 칸 | `name=` |
| **요청 사항** 텍스트박스 | `instructions=` (= 시스템 프롬프트) |
| 기본 도구 칸 | `tools=[]` (#8.2에서 `WebSearchTool` 추가) |

아래 셀 돌리면 `Agent` 객체가 찍힌다. **아직 아무것도 안 물어봤다** — 그냥 '코치를 정의'만 한 것. Gem 만들어놓고 채팅 안 친 상태랑 같다.

In [6]:
agent = Agent(
    name="Life Coach",
    instructions="너는 따뜻하고 격려해주는 라이프 코치야. 사용자의 말에 공감하고, 현실적이고 구체적인 조언을 한국어 존댓말로 해줘.",
)
agent

Agent(name='Life Coach', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='너는 따뜻하고 격려해주는 라이프 코치야. 사용자의 말에 공감하고, 현실적이고 구체적인 조언을 한국어 존댓말로 해줘.', prompt=None, handoffs=[], model=None, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=Reasoning(effort='none', generate_summary=None, summary=None), verbosity='low', metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

## 2) 메모리 만들기 = `SQLiteSession`

LLM은 통화 끊으면 잊는 글자 예측기다. 그래서 대화를 **밖(SQLite 파일)에** 쌓아두고, 매 호출마다 다시 넣어준다. (1주차 '리스트에 쌓기'의 프로덕션 버전 = 파일에 쌓기)

- `SQLiteSession(session_id, db_path)` → 대화가 `nb.sqlite` 파일에 자동 저장됨
- Streamlit에선 이걸 `st.session_state`에 캐싱한다(파일 재실행돼도 안 날아가게). 노트북은 셀 단위라 그냥 변수로 둔다.

In [4]:
session = SQLiteSession("life-coach-nb", "nb.sqlite")
session

## 3) 한 번 물어보기 — `await` 주의 (★ 노트의 그 함정)

`Runner.run(...)`은 **코루틴(async)**이다. 실행 방법이 환경마다 다르다:

- **주피터** = 이미 이벤트 루프가 돌고 있음 → `await`을 **바로** 쓴다 (아래)
- **Streamlit(.py)** = 동기 스크립트라 루프가 없음 → `asyncio.run(...)`으로 감싼다 (노트 #8.0의 그 줄)

> 둘이 같은 일(코루틴 실행)을 하는데 방식만 다른 것. 주피터에서 `asyncio.run()` 쓰면 'cannot be called from a running event loop' 에러 난다.

In [7]:
result = await Runner.run(agent, "안녕, 나 요즘 잠을 너무 못 자.", session=session)
print(result.final_output)

안녕하세요. 잠을 못 자시면 정말 힘드셨겠어요. 몸도 마음도 금방 지치기 쉽죠.

우선 오늘부터 바로 해볼 수 있는 것만 간단히 드릴게요:
- **잠들기 1시간 전**: 휴대폰, 영상, 자극적인 뉴스는 잠깐 멀리해보세요.
- **카페인**: 오후 늦게는 커피, 에너지음료를 피하시는 게 좋아요.
- **침대에서는 잠만**: 누워서 20분 넘게 못 자면 잠깐 일어나서 조용한 활동을 해보세요.
- **호흡**: 4초 들이마시고 6초 내쉬는 걸 5분만 해보세요.
- **생각 정리**: 걱정거리가 많다면 잠들기 전에 메모장에 적어두면 도움이 됩니다.

만약 **2주 이상** 계속되거나, 너무 자주 깨거나, 우울/불안이 심해지면 병원이나 수면클리닉 상담도 꼭 고려해보세요.

원하시면 제가 지금 **수면 습관 체크리스트**처럼 같이 원인 찾아드릴게요.


## 4) 기억하나? — 메모리 작동 증명

방금 '잠을 못 잔다'고 했다. 이번엔 그 말을 **안 반복**하고 물어본다. `session=session`을 같이 넘기니까, SDK가 이전 대화를 다시 LLM에 넣어준다 → 기억하는 것처럼 보인다.

(`session`을 안 넘기면 매번 처음 보는 사람 취급한다. 한번 빼고도 돌려보면 차이가 확 보인다.)

In [8]:
result = await Runner.run(agent, "내가 방금 무슨 고민이라고 했지?", session=session)
print(result.final_output)

잠을 너무 못 주무신다고 말씀하셨어요.


## 5) 메모리 속 '진짜 구조' 들여다보기 (★ #8.1 떡밥)

`session.get_items()`로 쌓인 메시지의 **날것 구조**를 본다. 여기가 #8.1 `paint_history`에서 헷갈리는 부분의 정체다:

- **user** 메시지: `content`가 그냥 **문자열**
- **assistant** 메시지: `type == 'message'`, `content`가 **리스트** → 텍스트는 `content[0]['text']`

왜 다르냐? assistant는 텍스트 말고 도구호출·이미지도 담을 수 있어서 '부품 리스트'로 온다. 직접 눈으로 확인:

In [9]:
items = await session.get_items()
print("총", len(items), "개 메시지\n")
for i, m in enumerate(items):
    print(f"--- [{i}] role={m.get('role')} type={m.get('type')} ---")
    print(m)
    print()

총 4 개 메시지

--- [0] role=user type=None ---
{'content': '안녕, 나 요즘 잠을 너무 못 자.', 'role': 'user'}

--- [1] role=assistant type=message ---
{'id': 'msg_0b0515727c35a2ac006a30d54caf888198a088021c6eb33c28', 'content': [{'annotations': [], 'text': '안녕하세요. 잠을 못 자시면 정말 힘드셨겠어요. 몸도 마음도 금방 지치기 쉽죠.\n\n우선 오늘부터 바로 해볼 수 있는 것만 간단히 드릴게요:\n- **잠들기 1시간 전**: 휴대폰, 영상, 자극적인 뉴스는 잠깐 멀리해보세요.\n- **카페인**: 오후 늦게는 커피, 에너지음료를 피하시는 게 좋아요.\n- **침대에서는 잠만**: 누워서 20분 넘게 못 자면 잠깐 일어나서 조용한 활동을 해보세요.\n- **호흡**: 4초 들이마시고 6초 내쉬는 걸 5분만 해보세요.\n- **생각 정리**: 걱정거리가 많다면 잠들기 전에 메모장에 적어두면 도움이 됩니다.\n\n만약 **2주 이상** 계속되거나, 너무 자주 깨거나, 우울/불안이 심해지면 병원이나 수면클리닉 상담도 꼭 고려해보세요.\n\n원하시면 제가 지금 **수면 습관 체크리스트**처럼 같이 원인 찾아드릴게요.', 'type': 'output_text', 'logprobs': []}], 'role': 'assistant', 'status': 'completed', 'type': 'message', 'phase': 'final_answer'}

--- [2] role=user type=None ---
{'content': '내가 방금 무슨 고민이라고 했지?', 'role': 'user'}

--- [3] role=assistant type=message ---
{'id': 'msg_0b0515727c35a2ac006a30d574a1d88198ada1c93f166a92c2', 'content'

## 6) 스트리밍 — 답이 글자 조각(delta)으로 온다

`Runner.run_streamed(...)`은 답을 다 기다리지 않고 **조각조각** 흘려준다 (어제 #7의 `run_streamed`).

- `event.type == 'raw_response_event'` → 저수준 모델 응답 (← 어제 퀴즈 Q8: 토큰 단위가 이것)
- 그 안에서 `event.data.type == 'response.output_text.delta'` → 텍스트 한 조각

아래 돌리면 글자가 **주르륵** 찍힌다. Streamlit에선 이 조각들을 `st.empty()` 한 칸에 **누적해서 덮어쓰기**(#8.1) → 매끈한 한 박스로 보인다.

In [11]:
runner = Runner.run_streamed(agent, "아침에 일찍 일어나는 현실적인 팁 3개만 짧게.", session=session)

async for event in runner.stream_events():
    if event.type == "raw_response_event":
        if event.data.type == "response.output_text.delta":
            print(event.data.delta, end="")

1. 알람을 손 닿지 않는 곳에 두세요.  
2. 일어나자마자 햇빛을 5분만 받으세요.  
3. 전날 밤에 “기상 후 첫 행동”을 정해두세요.

## 7) 그럼 Streamlit은 뭘 하나? (실행은 `app.py`에서)

여기까지가 **엔진** 전부다. 위 코드엔 화면이 없었다(`print`만). Streamlit은 이 엔진을 **채팅 화면에 붙이는 껍데기**일 뿐:

| 노트북에서 한 것 | Streamlit(app.py)에서 |
|---|---|
| `await Runner.run(...)` | `asyncio.run(run_agent(...))` (동기라 감쌈) |
| `print(delta, end="")` | `placeholder.write(누적문자열)` (`st.empty()`에 덮어쓰기) |
| `for m in items: print(m)` | `paint_history()` → `st.chat_message`로 그림 |
| 입력을 코드로 직접 | `prompt = st.chat_input(...)` |
| `session` 변수 | `st.session_state["session"]`에 캐싱 |

**Streamlit의 단 하나의 규칙**: 글자 칠 때마다 `app.py`를 처음부터 다시 실행한다 → 살아남아야 할 건(`session`, `agent`) `st.session_state`에, 매번 다시 그려야 할 건(대화 기록) `paint_history()`로.

→ 이건 `app.py`를 만들고 `uv run streamlit run app.py`로 띄워서 볼 것. (다음 단계)

## (선택) 메모리 리셋

처음부터 다시 실험하고 싶으면 메모리를 비운다. Streamlit '디버깅 사이드바'의 *Reset memory* 버튼이 하는 일이 이것.

In [ ]:
# await session.clear_session()  # 주석 풀고 실행하면 nb.sqlite 대화 전부 삭제
print("비우려면 윗줄 주석 해제")